## Issues
1. Capacity availability on certain date
   1. #of trucks available
2. Look at the transmode
   1. (Truck - Container) 40Ft truck  
3. Convert (shipment_plans) Units into pallets
4. Container should be limited by weight, area utilization, Volume
   1. Add-on: Conditions are Configurable from the user
   2. Calculate golden ratio (Currently A -> B)
5. Pull-in method


# Code

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import os

from objects.ContainerFleetOptimizationResult import ContainerFleetOptimizationResult
from objects.ContainerOptimizationResult import ContainerOptimizationResult
from pandas import DataFrame, merge, concat

from numpy import floor, ceil, cumsum, where
from collections import defaultdict
from logging import getLogger
import math, time

## Classes

from objects.Axle import Axle
from objects.Container import Container
from objects.ContainerSummary import ContainerSummary
from objects.ContainerLoadingRules import ContainerLoadingRules
from objects.Dimension import Dimension
from objects.Pallet import Pallet
from objects.Position import Position
import uuid

from objects.AxleLoadResult import AxleLoadResult
from objects.UtilizationMetrics import UtilizationMetrics
from objects.RemainingCapacity import RemainingCapacity


from __future__ import annotations
import math
import hashlib
from datetime import datetime
from typing import List, Optional, Tuple, Dict, Any
import pandas as pd
from objects.ShipmentGroup import ShipmentGroup
import json

##
from utils.visualize import container_visualization
from utils.measure_conversion import *

logger = getLogger("load_planner")


# Assumptions
1. Units in the pallet are homogeneous
2. Units are calculated from item quantity to pallets
3. Dimensions are measured in inches (Later converted to Feet / other metrics)
4. **Routes have been pre-planned**
5. 

# Solver
1. Decide what items to load based on:
- Delivery date, priority, Item name, 
- Convert units into pallets
- Confirm the #of units shipped & update 
2. Grouping logic:
- Combine all items

### Optimizers to look into
1. 
## Feature enhancements
1. Stock pull-in from future (Early shipping)
2. Axle based handling-unit 📦(container) positioning
3. 

In [3]:
### Writing results to Database ###
# Write results to tables:
# 1. Handling_unit
# 2. Handling_unit_content
# 3. Handling_unit_position
# 4. Transport_equipment_assignment
# 5. route_planned


In [4]:
### Verify loaded 🚚 truck_equipment_assignment & handling_unit positions inside the container  

## Establish connection with Neon Database

In [5]:
### NeonDB Connection
import database.helper as db_helper
db_conn = db_helper.create_connection()

## Loading Data
item_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.item_master", connection=db_conn)
lane_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.lane_master", connection=db_conn)
load_equipment_metadata_df = db_helper.fetch_data(sql="select * from inventory_management.public.load_equipment_metadata", connection=db_conn)
location_df = db_helper.fetch_data(sql="select * from inventory_management.public.location", connection=db_conn)
shipment_demand_df = db_helper.fetch_data(sql="select * from inventory_management.public.shipment_plans", connection=db_conn)
sku_uom_df = db_helper.fetch_data(sql="select * from inventory_management.public.sku_unit_of_measure", connection=db_conn)
transport_asset_df = db_helper.fetch_data(sql="select * from inventory_management.public.transport_asset", connection=db_conn)


D:\acies_solutions\solutions-inventory-optimization\database\helper.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


## Create necessary features, calculations




In [6]:
sku_uom_df = pd.concat(
    [
        sku_uom_df,
        sku_uom_df['pallet_dimensions'].apply(pd.Series)
    ],
    axis=1
)

sku_uom_column_mapper = {x:x for x in sku_uom_df.columns}
sku_uom_column_mapper['height_mm'] = 'pallet_height_mm'
sku_uom_column_mapper['width_mm'] = 'pallet_width_mm'
sku_uom_column_mapper['length_mm'] = 'pallet_length_mm'
sku_uom_df.rename(columns=sku_uom_column_mapper, inplace=True)

In [7]:

def create_pallet_features(
    shipment_demand_df: pd.DataFrame,
    sku_pallet_df: pd.DataFrame,           # columns from load_equipment_metadata or pallet_master
) -> pd.DataFrame:
    """
    Merges pallet/UOM master into shipment demand and computes:
      - required_pallets, full_pallets, remaining_units, partial_fill_pct
    Returns enriched shipment_candidate_df.
    """
    # Merge pallet dimensions onto demand rows
    candidate = shipment_demand_df.copy()

    # Columns expected from pallet_master: sku_id, unit_count_in_pallet,
    # pallet_height_mm, pallet_width_mm, pallet_length_mm,
    # pallet_weight_in_kg, item_weight_in_kg
    if "unit_count_in_pallet" not in candidate.columns:
        candidate = candidate.merge(
            sku_pallet_df[[
                "sku_id", "unit_count_in_pallet",
                "pallet_height_mm", "pallet_width_mm", "pallet_length_mm",
                "pallet_weight_in_kg", "item_weight_in_kg",
            ]],
            on="sku_id",
            how="left",
        )
 

    ### Filter for the ones that are divisible
    candidate = (
        candidate
        [
            candidate[
                "unit_count_in_pallet"
            ]
            > 0
        ]
    )
 
    # Compute pallet breakdown
    candidate["required_pallets"] = (
        candidate["planned_quantity"] / candidate["unit_count_in_pallet"]
    ).apply(math.ceil)
 
    candidate["full_pallets"] = (
        candidate["planned_quantity"] // candidate["unit_count_in_pallet"]
    ).astype(int)
 
    candidate["remaining_units"] = (
        candidate["planned_quantity"] % candidate["unit_count_in_pallet"]
    ).astype(int)
 
    candidate["partial_fill_pct"] = candidate.apply(
        lambda r: (r["remaining_units"] / r["unit_count_in_pallet"])
        if r["remaining_units"] > 0 else 0.0,
        axis=1,
    )
 
    return candidate
 


In [8]:
# shipment_candidate_df = (
#     create_pallet_features(
#         shipment_demand_df=shipment_demand_df,
#         sku_uom_df=sku_uom_df[['sku_id', 'unit_count_in_pallet', 'pallet_height_mm', 'pallet_width_mm', 'pallet_length_mm', 'pallet_weight_in_kg', 'item_weight_in_kg']],
#         delivery_date='2026-04-12'
#     )
# )

print('shipment_candidate_df', shipment_candidate_df.columns.to_list())

# Optimizer Flow V3

- STEP 1:
feature engineer

- STEP 2:
Placement engine

- STEP 3:
Container Fleet


## Feature Engineering

In [9]:

# ── Color palette for SKUs ────────────────────────────────────────────────────
_SKU_COLORS = [
    "#4CAF50", "#2196F3", "#FF9800", "#E91E63", "#9C27B0",
    "#00BCD4", "#FF5722", "#607D8B", "#795548", "#FFC107",
    "#3F51B5", "#8BC34A", "#F44336", "#009688", "#FFEB3B",
]



def _sku_color(sku_id: str) -> str:
    idx = int(hashlib.md5(sku_id.encode()).hexdigest(), 16) % len(_SKU_COLORS)
    return _SKU_COLORS[idx]


In [10]:

# ── Action 2: Break enriched demand rows into individual Pallet objects ───────
 
def breakdown_into_pallets(shipment_candidate_df: pd.DataFrame) -> List[Pallet]:
    """
    Explodes each demand row into N Pallet domain objects
    (full pallets + optional partial pallet).
    """
    pallets: List[Pallet] = []
    pallet_counter: Dict[str, int] = {}
 
    service_level_map = {"GOLD": 3, "SILVER": 2, "BRONZE": 1, "STANDARD": 1}
 
    for _, row in shipment_candidate_df.iterrows():
        base_id = f"{row['shipment_id']}_{row['sku_id']}"
        pallet_counter.setdefault(base_id, 0)
 
        dims = Dimension(
            depth=int(row["pallet_length_mm"]),
            width=int(row["pallet_width_mm"]),
            height=int(row["pallet_height_mm"]),
        )
        weight_per_pallet = (
            row["pallet_weight_in_kg"]
        )
        service_level_val = service_level_map.get(
            str(row.get("service_level", "STANDARD")).upper(), 1
        )
        est_date = (
            pd.to_datetime(row["estimated_delivery_date"])
            if "estimated_delivery_date" in row else datetime.utcnow()
        )
 
        color = _sku_color(str(row["sku_id"]))
        temperature_req = str(row.get("temperature_requirement", "ambient") or "ambient").lower()
        special = str(row.get("special_handling", "") or "").lower()
 
        def _make_pallet(seq: int, is_partial: bool, fill_pct: float, units: int) -> Pallet:
            pallet_counter[base_id] += 1
            pid = f"PLT_{row['shipment_id']}_{row['sku_id']}_{pallet_counter[base_id]:04d}"
            w = weight_per_pallet * fill_pct if is_partial else weight_per_pallet
            return Pallet(
                candidatePalletId=pid,
                shipmentId=str(row["shipment_id"]),
                skuId=str(row["sku_id"]),
                originLocationId=str(row["origin_location_id"]),
                destinationLocationId=str(row["destination_location_id"]),
                estimatedDeliveryDate=est_date,
                dimensions=dims,
                label=str(row.get("sku_id", pid)),
                color=color,
                weightIn_kg=round(w, 2),
                priority=int(row.get("priority", 0)),
                serviceLevel=service_level_val,
                unitsInPallet=units,
                isPartialPallet=is_partial,
                fillPct=fill_pct,
                unloadSequence=int(row.get("unload_sequence_preference", 0)),
                temperatureRequirement=temperature_req,
                isFragile="fragile" in special,
                isHazmat="hazmat" in special,
            )
 
        # Full pallets
        for i in range(int(row.get("full_pallets", 0))):
            pallets.append(_make_pallet(i, False, 1.0, int(row["unit_count_in_pallet"])))
 
        # Partial pallet
        if int(row.get("remaining_units", 0)) > 0:
            fill = row["partial_fill_pct"] if row["partial_fill_pct"] > 0 else (
                row["remaining_units"] / row["unit_count_in_pallet"]
            )
            pallets.append(_make_pallet(
                int(row.get("full_pallets", 0)),
                True,
                round(fill, 4),
                int(row["remaining_units"]),
            ))
 
    return pallets
 

In [11]:

# ── Action 3: Group pallets by shipment lane ──────────────────────────────────
 
def group_pallets_by_lane(
    pallets: List[Pallet],
    lane_master_df: pd.DataFrame | None = None,
    date_granularity: str = "day",   # "day" | "week"
) -> List[ShipmentGroup]:
    """
    Groups pallets by (origin, destination, delivery_date_window).
    Optionally enriches with lane metadata.
    """
    from collections import defaultdict
 
    bucket: Dict[Tuple, List[Pallet]] = defaultdict(list)
 
    for p in pallets:
        if date_granularity == "week":
            dt_key = p.estimatedDeliveryDate.strftime("%Y-W%W")
        else:
            dt_key = p.estimatedDeliveryDate.strftime("%Y-%m-%d")
 
        key = (p.originLocationId, p.destinationLocationId, dt_key)
        bucket[key].append(p)
 
    groups: List[ShipmentGroup] = []
    for idx, ((origin, dest, dt_key), pallet_list) in enumerate(bucket.items()):
        group_id = f"GRP_{origin}_{dest}_{dt_key}".replace(" ", "_").replace("-", "")
        groups.append(ShipmentGroup(
            groupId=group_id,
            originLocationId=origin,
            destinationLocationId=dest,
            deliveryDateWindow=dt_key,
            pallets=pallet_list,
            estimatedDeliveryDate=pallet_list[0].estimatedDeliveryDate,
        ))
 
    return groups
 
 

In [12]:

# ── Action 4: Sort pallets for optimal loading sequence ───────────────────────
 
def sort_pallets_for_loading(pallets: List[Pallet], lifo: bool = True) -> List[Pallet]:
    """
    Sort order:
      1. destinationStop ASC  (multi-drop: last destination loaded first for LIFO)
      2. priority DESC
      3. serviceLevel DESC
      4. estimatedDeliveryDate ASC
      5. full pallet before partial (isPartialPallet ASC)
      6. heavier pallets first (better CoG)
 
    When LIFO is enabled, pallets for the last stop are loaded first
    (they will be unloaded last = deepest in container).
    """
    # Determine stop ordering: reverse for LIFO
    dest_order_sign = -1 if lifo else 1
 
    return sorted(
        pallets,
        key=lambda p: (
            dest_order_sign * p.destinationStop,   # LIFO: last stop loads first
            -p.priority,
            -p.serviceLevel,
            p.estimatedDeliveryDate,
            int(p.isPartialPallet),
            -p.weightIn_kg,
        ),
    )
 

## Placement Engine


Placement Engine
================
Advanced 3D bin-packing for pallets into containers.
 
Algorithm: Skyline / Column-Strip with extreme-point extension.
  - Tracks a "skyline" of occupied depth per width-strip column.
  - For each pallet: tries all valid orientations, scores positions
    by (contact surface, CoG improvement, axle balance), picks best.
  - Stacking supported optionally.
 
Actions:
  - validate_pallet_fits(container, pallet) -> (bool, str)
  - find_best_position(container, pallet) -> Position | None
  - place_pallet(container, pallet) -> bool
  - compute_axle_loads(container) -> List[AxleLoadResult]
  - compute_utilization(container, pending) -> (UtilizationMetrics, RemainingCapacity)

In [13]:

# ── Orientation helpers ───────────────────────────────────────────────────────
 
def _orientations(dim: Dimension) -> List[Tuple[int, int, int, str]]:
    """
    Returns (depth, width, height, orientation_label) for each valid rotation.
    We only allow 90° rotations around vertical axis (pallets stay upright).
    """
    d, w, h = dim.depth, dim.width, dim.height
    return [
        (d, w, h, "FRONT_FACING"),
        (w, d, h, "SIDE_FACING_LEFT"),
    ]
 


In [14]:
"""
_Skyline — with 50 mm inter-pallet spacing
===========================================

Spacing design
--------------
GAP = PALLET_GAP_MM (default 50 mm) is applied on BOTH axes:

  Depth axis (z):
    mark_occupied() advances the skyline to (z + depth + GAP).
    The next pallet in the same column therefore starts at least GAP mm
    clear of the previous one's trailing edge.

  Width axis (x):
    _overlaps() treats every placed block as [bx .. bx+bw+GAP) × [bz .. bz+bd+GAP).
    A column-2 candidate at x = bx+bw+GAP is exactly flush with that boundary
    (x < bx+bw+GAP  →  1269 < 1269 = False) so it passes overlap cleanly.
    The skyline columns are NOT extended by GAP on the width axis — doing so
    would bleed the skyline height into column-2 strips and incorrectly block z=0.

Stored coordinates
------------------
position.x, position.z are the TRUE pallet corner — no phantom offset.
effectiveWidth / effectiveDepth are the TRUE placed footprint.
React renders the actual box; the gap lives only in the reservation logic.

Two-column layout
-----------------
After P1 at (x=0, z=0, wid=W1):
  - candidate (W1+GAP, 0) is generated from the block extreme-point rule
  - _overlaps(W1+GAP, 0, ...) is False  ← x=W1+GAP is NOT < bx+bw+GAP=W1+GAP
  - skyline[col covering W1+GAP] = 0    ← untouched, so z=0 passes min_z check
  ⇒ column 2 pallet places flush against the gap at x=W1+GAP, z=0  ✓
"""

import math
from typing import List, Tuple

PALLET_GAP_MM: int = 50  # change here to adjust spacing globally


class _Skyline:
    def __init__(
        self,
        container_width: int,
        container_depth: int,
        resolution: int = 50,
        gap_mm: int = PALLET_GAP_MM,
    ):
        self.resolution      = resolution
        self.gap             = gap_mm
        self.n_cols          = math.ceil(container_width / resolution)
        self.container_width = container_width
        self.container_depth = container_depth
        # skyline[c] = deepest reserved z in column c (door = z=0, back = z=max)
        self.skyline: List[int] = [0] * self.n_cols
        # Stored with TRUE placed dims  (x, y, z, actual_depth, actual_width, height)
        self.blocks: List[Tuple[int, int, int, int, int, int]] = []

    # ── Column helpers ─────────────────────────────────────────────────────

    def _col_range(self, x_mm: int, w_mm: int) -> Tuple[int, int]:
        c0 = x_mm // self.resolution
        c1 = math.ceil((x_mm + w_mm) / self.resolution)
        return max(0, c0), min(self.n_cols, c1)

    def max_depth_in_strip(self, x_mm: int, w_mm: int) -> int:
        c0, c1 = self._col_range(x_mm, w_mm)
        return max(self.skyline[c0:c1]) if c0 < c1 else 0

    # ── Mark occupied ──────────────────────────────────────────────────────

    def mark_occupied(
        self,
        x_mm: int, z_mm: int,
        d_mm: int, w_mm: int, h_mm: int,
    ) -> None:
        """
        Reserve the pallet's depth footprint + GAP on the skyline.

        Only the pallet's actual width columns are updated (no width bleed).
        Width-axis spacing is enforced entirely through _overlaps().
        """
        # Push skyline to z + depth + GAP so next row starts GAP mm clear
        reserved_z = min(z_mm + d_mm + self.gap, self.container_depth)
        c0, c1 = self._col_range(x_mm, w_mm)   # actual width columns only
        for c in range(c0, c1):
            self.skyline[c] = max(self.skyline[c], reserved_z)
        # Store true placed dims (no gap baked in)
        self.blocks.append((x_mm, 0, z_mm, d_mm, w_mm, h_mm))

    # ── Candidate positions ────────────────────────────────────────────────

    def candidate_positions(
        self, pallet_depth: int, pallet_width: int
    ) -> List[Tuple[int, int]]:
        """
        Returns valid (x, z) placement corners with 50 mm spacing baked in.

        Candidates are generated at:
          (bx,           bz + bd + GAP)  — next row, same column
          (bx + bw + GAP, bz)            — column 2, same row
          (bx + bw + GAP, bz + bd + GAP) — diagonal
          skyline step transitions (with and without GAP offset)
        """
        positions: set[Tuple[int, int]] = set()
        g = self.gap

        positions.add((0, 0))  # first pallet (door end / front of container)

        for (bx, by, bz, bd, bw, bh, *_) in self.blocks:
            positions.add((bx,          bz + bd + g))   # next row (depth axis)
            positions.add((bx + bw + g, bz))            # column 2 (width axis)
            positions.add((bx + bw + g, bz + bd + g))   # diagonal corner

        # Skyline transitions: seam between different-height strips
        for c in range(self.n_cols - 1):
            if self.skyline[c] != self.skyline[c + 1]:
                x_snap = c * self.resolution
                positions.add((x_snap,      self.skyline[c]))
                positions.add((x_snap,      self.skyline[c + 1]))
                positions.add((x_snap + g,  self.skyline[c]))   # gap-shifted seam

        cw = self.container_width
        cd = self.container_depth
        valid: List[Tuple[int, int]] = []

        for (x, z) in sorted(positions):  # deterministic: ascending x then z
            x, z = int(x), int(z)
            if x < 0 or z < 0:
                continue
            if x + pallet_width > cw:     # pallet must fit within interior
                continue
            if z + pallet_depth > cd:
                continue
            # z must be at or above the skyline for all spanned columns
            c0, c1 = self._col_range(x, pallet_width)
            min_z_needed = max(self.skyline[c0:c1]) if c0 < c1 else 0
            if z < min_z_needed:
                continue
            # Gap-aware overlap check
            if not self._overlaps(x, z, pallet_depth, pallet_width):
                valid.append((x, z))

        return valid

    # ── Overlap check ──────────────────────────────────────────────────────

    def _overlaps(self, x: int, z: int, d: int, w: int) -> bool:
        """
        AABB overlap against the GAP-expanded block footprint.

        Each placed block is treated as occupying:
          x-axis: [bx .. bx + bw + GAP)
          z-axis: [bz .. bz + bd + GAP)

        Flush contact (new pallet starts exactly at bx+bw+GAP or bz+bd+GAP)
        evaluates as  x < bx+bw+GAP → x < x → False  → no overlap.
        This guarantees exactly GAP mm of clear space between all neighbours.
        """
        g = self.gap
        for (bx, by, bz, bd, bw, bh, *_) in self.blocks:
            if (x     < bx + bw + g and
                x + w > bx          and
                z     < bz + bd + g and
                z + d > bz):
                return True
        return False

In [15]:

# ── Main validation ───────────────────────────────────────────────────────────
 
def validate_pallet_fits(container: Container, pallet: Pallet) -> Tuple[bool, str]:
    """Check weight, volume, floor area, temperature, and hazmat constraints."""
    # Weight
    if container.usedWeightIn_kg + pallet.weightIn_kg > container.maxPayloadWeightIn_kg:
        return False, (
            f"Weight limit exceeded: "
            f"{container.usedWeightIn_kg + pallet.weightIn_kg:.1f} > "
            f"{container.maxPayloadWeightIn_kg:.1f} kg"
        )
    # Volume
    if container.usedVolume_m3 + pallet.volume_m3 > container.maxVolume_m3:
        return False, (
            f"Volume limit exceeded: "
            f"{container.usedVolume_m3 + pallet.volume_m3:.3f} > "
            f"{container.maxVolume_m3:.3f} m³"
        )
    # Floor area
    if container.usedFloorArea_m2 + pallet.floorArea_m2 > container.maxFloorArea_m2:
        return False, (
            f"Floor area limit exceeded: "
            f"{container.usedFloorArea_m2 + pallet.floorArea_m2:.2f} > "
            f"{container.maxFloorArea_m2:.2f} m²"
        )
    # Temperature
    if pallet.temperatureRequirement not in ("ambient", ""):
        if not container.refrigerationCapable:
            return False, "Refrigeration required but container not capable"
    # Hazmat
    if pallet.isHazmat and container.loadingRules.hazmatSegregation:
        has_non_hazmat = any(not p.isHazmat for p in container.pallets)
        if has_non_hazmat:
            return False, "Hazmat segregation violation"
 
    return True, ""

In [16]:

# ── Scoring ───────────────────────────────────────────────────────────────────
 
def _score_position(
    x: int, z: int, d: int, w: int, h: int,
    container: Container,
    skyline: _Skyline,
    pallet_weight: float,
) -> float:
    """
    Score a placement candidate. Higher = better.
    Factors:
      - Compactness: favour positions closer to door (lower z)
      - Contact surface: favour touching walls / other pallets
      - CoG balance: favour centred x
      - Axle balance: favour positions that help front/rear balance
    """
    score = 0.0
 
    # Compactness (prefer loading from the back = high z for LIFO)
    score -= z * 0.001  # slight preference for deeper placement
 
    # Centred width (better stability)
    centre_x = container.internalWidth / 2
    deviation = abs((x + w / 2) - centre_x) / container.internalWidth
    score -= deviation * 5
 
    # Wall / neighbour contact bonus
    if x == 0 or x + w >= container.internalWidth:
        score += 2
    if z == 0:
        score += 1
 
    # Axle weight balance: prefer centred depth
    depth_ratio = (z + d / 2) / container.internalDepth
    score -= abs(depth_ratio - 0.5) * 3
 
    return score
 

In [17]:

def _distribute_weight_to_axles(
    container: Container, pallet: Pallet, z_mm: int, depth_mm: int
) -> None:
    """
    Distribute pallet weight across axles based on longitudinal position.
    Simple beam formula: load on axle i proportional to proximity.
    """
    pallet_cog_z = z_mm + depth_mm / 2  # CoG from door
    total_len = container.internalDepth
 
    for axle in container.axles:
        # Proximity factor: 1 when directly over axle, 0 at the other end
        dist = abs(pallet_cog_z - axle.positionX)
        factor = max(0.0, 1 - dist / total_len)
        axle.currentLoad += pallet.weightIn_kg * factor
 
 
# ── Axle load analysis ────────────────────────────────────────────────────────
 
def compute_axle_loads(container: Container) -> List[AxleLoadResult]:
    results = []
    for axle in container.axles:
        util = (axle.currentLoad / axle.maxWeight * 100) if axle.maxWeight > 0 else 0
        results.append(AxleLoadResult(
            axleId=axle.axleId,
            currentLoad_kg=round(axle.currentLoad, 2),
            maxLoad_kg=axle.maxWeight,
            utilization_pct=round(util, 2),
            isOverloaded=axle.currentLoad > axle.maxWeight,
        ))
    return results
 

In [18]:

# ── Position finder ───────────────────────────────────────────────────────────
 
def find_best_position(
    container: Container,
    pallet: Pallet,
    skyline: _Skyline,
) -> Optional[Position]:
    """
    Try all orientations × candidate positions, return best Position or None.
    """
    iW = int(container.internalWidth)
    iD = int(container.internalDepth)
    iH = int(container.internalHeight)
 
    best_pos: Optional[Position] = None
    best_score = float("-inf")
 
    for (dep, wid, hgt, orient) in _orientations(pallet.dimensions):
        # Height check
        if hgt > iH:
            continue
        # Door width / height check for first pallet
        if dep > container.doorHeight or wid > container.doorWidth:
            pass  # still ok if not first; door constraint mainly for forklift entry
 
        candidates = skyline.candidate_positions(dep, wid)
        if not candidates:
            # Fallback: place at current depth position
            candidates = [(0, skyline.max_depth_in_strip(0, iW))]
 
        for (x, z) in candidates:
            # Boundary checks
            if x + wid > iW or z + dep > iD:
                continue
            # Overlap check already done in skyline, double-check height
            score = _score_position(x, z, dep, wid, hgt, container, skyline, pallet.weightIn_kg)
            if score > best_score:
                best_score = score
                best_pos = Position(
                    x=x, y=0, z=z,
                    orientation=orient,
                    effectiveWidth=wid,   # actual x-footprint after rotation
                    effectiveDepth=dep,   # actual z-footprint after rotation
                    effectiveHeight=hgt,  # height unchanged
                )
                # Also stash on __dict__ for use in place_pallet mark_occupied call
                best_pos.__dict__["_dep"] = dep
                best_pos.__dict__["_wid"] = wid
                best_pos.__dict__["_hgt"] = hgt
 
    return best_pos
 

In [19]:
 
# ---------------------------------------------------------------------------
# Free rectangle dataclass
# ---------------------------------------------------------------------------
 
from dataclasses import dataclass


@dataclass
class FreeRect:
    x: int   # left edge  (width axis)
    z: int   # front edge (depth axis, 0 = door)
    w: int   # width extent
    d: int   # depth extent
 
    def contains(self, other: "FreeRect") -> bool:
        return (
            other.x >= self.x and other.z >= self.z and
            other.x + other.w <= self.x + self.w and
            other.z + other.d <= self.z + self.d
        )
 

In [20]:

# ── Place pallet action ───────────────────────────────────────────────────────
 
# Module-level skylines keyed by container id
_skylines: dict[str, _Skyline] = {}
 
 
def _get_skyline(container: Container) -> _Skyline:
    if container.containerId not in _skylines:
        _skylines[container.containerId] = _Skyline(
            int(container.internalWidth), int(container.internalDepth)
        )
    return _skylines[container.containerId]
 
 
def reset_skyline(container_id: str) -> None:
    _skylines.pop(container_id, None)
 
 
def place_pallet(container: Container, pallet: Pallet) -> bool:
    """
    Attempt to place pallet into container.
    Mutates container state and pallet.position on success.
    Returns True if placed, False otherwise.
    """
    ok, reason = validate_pallet_fits(container, pallet)
    if not ok:
        pallet.rejectionReason = reason
        return False
 
    skyline = _get_skyline(container)
    pos = find_best_position(container, pallet, skyline)
 
    if pos is None:
        pallet.rejectionReason = "No valid position found in container"
        return False
 
    # Extract chosen dims
    dep = pos.__dict__.get("_dep", pallet.dimensions.depth)
    wid = pos.__dict__.get("_wid", pallet.dimensions.width)
    hgt = pos.__dict__.get("_hgt", pallet.dimensions.height)
 
    # Mark position
    pallet.position = pos
    pallet.loadedToContainer = True
 
    # Update container state
    container.usedWeightIn_kg += pallet.weightIn_kg
    container.usedFloorArea_m2 += pallet.floorArea_m2
    container.usedVolume_m3 += pallet.volume_m3
    container.loadedPallets += 1
 
    # Mark skyline
    skyline.mark_occupied(pos.x, pos.z, dep, wid, hgt)
 
    # Update axle loads
    _distribute_weight_to_axles(container, pallet, pos.z, dep)
 
    return True
 

In [21]:

# ── Utilization ───────────────────────────────────────────────────────────────
 
def compute_utilization(
    container: Container, pending: List[Pallet]
) -> Tuple[UtilizationMetrics, RemainingCapacity]:
    wt_util = (container.usedWeightIn_kg / container.maxPayloadWeightIn_kg * 100
               if container.maxPayloadWeightIn_kg else 0)
    vol_util = (container.usedVolume_m3 / container.maxVolume_m3 * 100
                if container.maxVolume_m3 else 0)
    area_util = (container.usedFloorArea_m2 / container.maxFloorArea_m2 * 100
                 if container.maxFloorArea_m2 else 0)
 
    util = UtilizationMetrics(
        weightUtilization_pct=round(wt_util, 2),
        volumeUtilization_pct=round(vol_util, 2),
        floorAreaUtilization_pct=round(area_util, 2),
        loadedPallets=container.loadedPallets,
        totalPallets=container.loadedPallets + len(pending),
        usedWeightIn_kg=round(container.usedWeightIn_kg, 2),
        usedVolumeIn_m3=round(container.usedVolume_m3, 4),
        usedFloorAreaIn_m2=round(container.usedFloorArea_m2, 4),
        maxWeightIn_kg=container.maxPayloadWeightIn_kg,
        maxVolumeIn_m3=round(container.maxVolume_m3, 4),
        maxFloorAreaIn_m2=round(container.maxFloorArea_m2, 4),
    )
    remaining = RemainingCapacity(
        remainingWeight_kg=round(container.maxPayloadWeightIn_kg - container.usedWeightIn_kg, 2),
        remainingVolume_m3=round(container.maxVolume_m3 - container.usedVolume_m3, 4),
        remainingFloorArea_m2=round(container.maxFloorArea_m2 - container.usedFloorArea_m2, 4),
    )
    return util, remaining

## Container Fleet Optimizer

Orchestrates the full load planning pipeline:
Actions (function calls):
 - load_equipment_to_container_spec(row)
 - open_new_container(spec, container_idx, group)
 - load_pallets_into_container(container, pallets) -> ContainerOptimizationResult
 - optimize_container_fleet(group, equipment_df, rules) -> ContainerFleetOptimizationResult
 - run_full_optimization(shipment_candidate_df, ...) -> ContainerFleetOptimizationResult
 - export_container_json(result, out_dir) -> str

In [22]:

 
# ── Action: Map equipment row → Container spec ────────────────────────────────
 
def load_equipment_to_container_spec(row: pd.Series) -> Dict[str, Any]:
    """Convert a load_equipment_metadata_df row to Container constructor kwargs."""
    axle_conf = str(row.get("axle_configuration", "single")).lower()
    axles = []
    if "tandem" in axle_conf or "2" in axle_conf:
        axles = [
            Axle(axleId="FRONT", maxWeight=10000, positionX=float(row.get("internal_length_mm", 12000)) * 0.15),
            Axle(axleId="REAR", maxWeight=20000, positionX=float(row.get("internal_length_mm", 12000)) * 0.75),
        ]
    else:
        axles = [Axle(axleId="DEFAULT", maxWeight=30000,
                       positionX=float(row.get("internal_length_mm", 12000)) * 0.45)]
 
    return dict(
        containerType=str(row.get("equipment_name", "CONTAINER")),
        containerDepth=float(row.get("length_mm", 12191)),
        containerWidth=float(row.get("width_mm", 2438)),
        containerHeight=float(row.get("height_mm", 2591)),
        internalDepth=float(row.get("internal_length_mm", 11836)),
        internalWidth=float(row.get("internal_width_mm", 2352)),
        internalHeight=float(row.get("internal_height_mm", 2391)),
        maxPayloadWeightIn_kg=float(row.get("max_payload_weight_kg", 25000)),
        tareWeightIn_kg=float(row.get("tare_weight_kg", 2000)),
        doorWidth=float(row.get("door_width_mm", 2352)),
        doorHeight=float(row.get("door_height_mm", 2391)),
        refrigerationCapable=bool(row.get("refrigeration_capable", False)),
        temperatureMin_c=row.get("temperature_min_c"),
        temperatureMax_c=row.get("temperature_max_c"),
        axles=axles,
        loadingRules=ContainerLoadingRules(
            maxStackHeightIn_mm=float(row.get("max_stack_height_mm", 0)) or 0,
            allowStacking=float(row.get("max_stack_height_mm", 0) or 0) > 0,
        ),
    )
 

In [23]:

# ── Action: Open a fresh container for a group ────────────────────────────────
 
def open_new_container(
    spec: Dict[str, Any],
    container_idx: int,
    group: ShipmentGroup,
) -> Container:
    """Instantiate a new Container with summary pre-populated from the lane group."""
    cid = f"CONT_{group.originLocationId}_{group.destinationLocationId}_{container_idx:03d}"
    summary = ContainerSummary(
        routeId=group.groupId,
        origin=group.originLocationId,
        destinationInSequence=[group.destinationLocationId],
    )
    container = Container(
        containerId=cid,
        pallets=[],
        summary=summary,
        **spec,
    )
    return container
 

In [24]:

# ── Action: Load sorted pallets into a single container ───────────────────────
 
def load_pallets_into_container(
    container: Container,
    pallets: List[Pallet],
) -> ContainerOptimizationResult:
    """
    Greedily loads pallets into container until no more fit.
    Returns a ContainerOptimizationResult with loaded / pending pallets.
    """
    reset_skyline(container.containerId)
    loaded: List[Pallet] = []
    pending: List[Pallet] = []
 
    for pallet in pallets:
        if place_pallet(container, pallet):
            container.pallets.append(pallet)
            loaded.append(pallet)
        else:
            pending.append(pallet)
 
    # Update container summary
    container.summary.totalPallets = len(loaded)
    container.summary.totalWeightIn_kg = round(container.usedWeightIn_kg, 2)
    container.summary.totalVolumeIn_m3 = round(container.usedVolume_m3, 4)
 
    axle_loads = compute_axle_loads(container)
    utilization, remaining = compute_utilization(container, pending)
 
    return ContainerOptimizationResult(
        container=container,
        loadedPallets=loaded,
        pendingPallets=pending,
        axleLoads=axle_loads,
        utilization=utilization,
        remainingCapacity=remaining,
    )
 
 

In [25]:

# ── Action: Optimize full fleet for a shipment group ─────────────────────────
 
def optimize_container_fleet(
    group: ShipmentGroup,
    equipment_spec: Dict[str, Any],
    lifo: bool = True,
    max_containers: int = 1,
) -> ContainerFleetOptimizationResult:
    """
    Creates as many containers as needed to load all pallets in the group.
    Current version: infinite containers available.
    """
    sorted_pallets = sort_pallets_for_loading(group.pallets, lifo=lifo)
    remaining = list(sorted_pallets)
 
    all_containers: List[Container] = []
    all_results: List[ContainerOptimizationResult] = []
    container_idx = 1
 
    while remaining and container_idx <= max_containers:
        container = open_new_container(equipment_spec, container_idx, group)
        result = load_pallets_into_container(container, remaining)
 
        all_containers.append(container)
        all_results.append(result)
 
        # Pallets still pending after this container
        remaining = result.pendingPallets
 
        if not result.loadedPallets:
            # No pallet fit at all — break to avoid infinite loop
            break
 
        container_idx += 1
 
    # Fleet metrics
    total_wt = sum(c.usedWeightIn_kg for c in all_containers)
    total_max_wt = sum(c.maxPayloadWeightIn_kg for c in all_containers)
    total_vol = sum(c.usedVolume_m3 for c in all_containers)
    total_max_vol = sum(c.maxVolume_m3 for c in all_containers)
 
    return ContainerFleetOptimizationResult(
        containers=all_containers,
        containerResults=all_results,
        unallocated_pallets=remaining,
        total_pallets=len(group.pallets),
        total_containers=len(all_containers),
        fleet_weight_utilization_pct=round(total_wt / total_max_wt * 100, 2) if total_max_wt else 0,
        fleet_volume_utilization_pct=round(total_vol / total_max_vol * 100, 2) if total_max_vol else 0,
        optimizer_run_id=uuid.uuid4().hex,
    )
 



In [26]:

# ── Action: Full pipeline entry point ─────────────────────────────────────────
 
def run_full_optimization(
    shipment_demand_df: pd.DataFrame,
    sku_pallet_df: pd.DataFrame,
    load_equipment_metadata_df: pd.DataFrame,
    lane_master_df: Optional[pd.DataFrame] = None,
    preferred_equipment_type: str = "CONTAINER",
    lifo: bool = True,
) -> Dict[str, ContainerFleetOptimizationResult]:
    """
    End-to-end optimization:
      1. Feature engineering → pallets
      2. Group by lane
      3. Per group: fleet optimization
    Returns dict keyed by groupId.
    """
    # Step 1 — Enrich demand with pallet features
    candidate_df = create_pallet_features(shipment_demand_df, sku_pallet_df)
 
    # Step 2 — Breakdown into Pallet objects
    all_pallets = breakdown_into_pallets(candidate_df)
 
    # Step 3 — Group by lane
    groups = group_pallets_by_lane(all_pallets, lane_master_df)
 
    # Step 4 — Select default equipment spec
    eq_row = load_equipment_metadata_df[
        load_equipment_metadata_df["equipment_type"].str.upper() == preferred_equipment_type.upper()
    ].iloc[0] if len(load_equipment_metadata_df) else pd.Series()
 
    if eq_row.empty:
        eq_row = load_equipment_metadata_df.iloc[0]
 
    equipment_spec = load_equipment_to_container_spec(eq_row)
 
    # Step 5 — Optimize each group
    results: Dict[str, ContainerFleetOptimizationResult] = {}
    for group in groups:
        fleet_result = optimize_container_fleet(
            group, equipment_spec, lifo=lifo
        )
        results[group.groupId] = fleet_result
 
    return results
 

In [27]:

# ── Action: Export to JSON ────────────────────────────────────────────────────
 
def _pallet_to_viz_dict(p: Pallet) -> dict:
    return {
        "dimensions": {
            "depth": p.dimensions.depth,
            "width": p.dimensions.width,
            "height": p.dimensions.height,
        },
        "position": {
            "x": p.position.x,
            "y": p.position.y,
            "z": p.position.z,
        },
        "label": p.label or p.skuId,
        "color": p.color,
        "weightIn_kg": p.weightIn_kg,
        "isPartialPallet": p.isPartialPallet,
        "fillPct": p.fillPct,
        "shipmentId": p.shipmentId,
        "skuId": p.skuId,
        "priority": p.priority,
        "destinationStop": p.destinationStop,
        "unloadSequence": p.unloadSequence,
    }
 

In [28]:

from utils.CustomJSONEncoder import CustomJSONEncoder


def export_container_json(
    fleet_result: ContainerFleetOptimizationResult,
    out_dir: str = "./output",
    group_id: str = "default",
) -> List[str]:
    """
    Dumps each container as a visualization-ready JSON file.
    Returns list of file paths written.
    """
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    
    for cr in fleet_result.containerResults:
        c = cr.container

        fname = f"{group_id}_{c.containerId}.json"
        fpath = os.path.join(out_dir, fname)
        with open(fpath, "w") as f:
            json.dump(c, f, indent=2, cls=CustomJSONEncoder)
        paths.append(fpath)

    return paths
 

In [ ]:
result = run_full_optimization(
    shipment_demand_df=shipment_demand_df,
    sku_pallet_df=sku_uom_df,
    load_equipment_metadata_df=load_equipment_metadata_df,
    lane_master_df=lane_master_df,
    preferred_equipment_type='CONTAINER',
    fleet_limit=10,# Truck & Container limit               # 10 trucks total, all groups
    avg_pallets_per_container=20, # tune to your actual pallet density
    lifo=True,
)

In [30]:
for k in result.keys():
    print(result[k].total_pallets)
    if result[k].total_pallets>0:
        export_container_json(result[k])

12
30
30
34
14
176
27
47
21
143
52
22
69
240
1
269
1047
620
964
102
169
655
472
109
81
28
132
25
233
165
2580
629
596
686
619
667
486
107
524
1516
735
570
235
1209
3327
937
3448
4249
36
55
54
150
72
118
48
54
50
97
129
47
1
1
285
352
323
15
7
57
73
12
88
63
363
31
41
11
1
22
24
67
202
46
411
79
40
73
185
56
83
45
383
46
105
38
58
39
1
14
105
54
100
16
1
51
101
52
10
3
36
32
4
40
36
2
2
3
1
3
2
1
2
4
2
2
375
291
12
71
331
26
30
14
119
35
13
50
527
13
26
3
2
3
2
4
2
3
4
2
2
4
1
3
5
2
1
8
2
15
1
1
1
1
1
2
75
113
54
215
254
230
30
78
89
203
47
75
58
61
69
61
56
183
494
190
626
32
641
175
80
128
164
273
772
26
92
186
1026
240
111
19
8
5
44
30
24
35
58
25
51
105
363
1475
290
1707
623
344
110
1941
197
214
1562
68
403
89
120
130
107
69
1
1
1
1
1
1
1
1
1
2
2
1
1
2
2
1
49
121
72
1
18
88
19
1
23
21
23
13
7
1
2
2
1
1
3
2
3
1
2
193
180
971
616
368
50
25
510
62
121
158
190
120
447
120
213
96
39
23
9
50
84
13
48
95
44
101
33
24
58
1
9
22
3
3
22
24
12
90
28
27
53
51
56
14
70
121
9
9
14
16
16
39
45
14


In [31]:
result['GRP_6010_6048_20260607'].total_pallets

2

In [32]:
candidate_df = create_pallet_features(shipment_demand_df, sku_uom_df)

# Step 2 — Breakdown into Pallet objects
all_pallets = breakdown_into_pallets(candidate_df)



In [33]:
candidate_df

,order_line_id,shipment_id,sku_id,actual_delivery_date,origin_location_id,destination_location_id,estimated_delivery_date,planned_quantity,shipped_quantity,weight_kg,...,unit_count_in_pallet,pallet_height_mm,pallet_width_mm,pallet_length_mm,pallet_weight_in_kg,item_weight_in_kg,required_pallets,full_pallets,remaining_units,partial_fill_pct
0,23654,Lane_From_6011_To_0848-0001-140201-13-Apr-26-3,140201,None,6011,0848,2026-04-13,4320,0,1675387.41,...,2160.0,1155.0,1016.0,1219.0,387.82,0.18,2,2,0,0.000000
1,23655,Lane_From_6011_To_0848-0001-128489-12-Apr-26-1,128489,None,6011,0848,2026-04-12,12960,0,6924934.63,...,720.0,1190.0,1016.0,1219.0,534.33,0.74,18,18,0,0.000000
2,23656,Lane_From_6011_To_0848-0001-128472-14-Apr-26-1,128472,None,6011,0848,2026-04-14,36720,0,13574557.07,...,2160.0,1057.0,1016.0,1219.0,369.68,0.17,17,17,0,0.000000
3,23657,Lane_From_6011_To_0848-0001-140249-14-Apr-26-1,140249,None,6011,0848,2026-04-14,28080,0,10890018.17,...,2160.0,1155.0,1016.0,1219.0,387.82,0.18,13,13,0,0.000000
4,23658,Lane_From_6011_To_0848-0001-128685-15-Apr-26-1,128685,None,6011,0848,2026-04-15,17280,0,9233246.18,...,720.0,1190.0,1016.0,1219.0,534.33,0.74,24,24,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23648,47302,Lane_From_6119_To_6073-0001-194494-09-Jun-26-6,194494,None,6119,6073,2026-06-09,81,0,71093.74,...,3630.0,1397.0,1016.0,1219.0,877.70,0.24,1,0,81,0.022314
23649,47303,Lane_From_6119_To_6073-0001-194498-29-Jun-26-0,194498,None,6119,6073,2026-06-29,2066,0,1813329.27,...,3630.0,1397.0,1016.0,1219.0,877.70,0.24,1,0,2066,0.569146
23650,47304,Lane_From_6119_To_6073-0001-194498-29-Jun-26-7,194498,None,6119,6073,2026-06-29,1564,0,1372723.61,...,3630.0,1397.0,1016.0,1219.0,877.70,0.24,1,0,1564,0.430854
23651,47305,Lane_From_6119_To_6073-0001-189231-21-Apr-26-6,189231,None,6119,6073,2026-04-21,930,0,855070.82,...,1344.0,1333.0,1016.0,1219.0,919.43,0.68,1,0,930,0.691964


In [34]:
all_pallets[0]

Pallet(candidatePalletId='PLT_Lane_From_6011_To_0848-0001-140201-13-Apr-26-3_140201_0001', shipmentId='Lane_From_6011_To_0848-0001-140201-13-Apr-26-3', skuId='140201', originLocationId='6011', destinationLocationId='0848', estimatedDeliveryDate=Timestamp('2026-04-13 00:00:00'), dimensions=Dimension(depth=1219, width=1016, height=1155), position=Position(x=-1, y=-1, z=-1, orientation='FRONT_FACING', effectiveWidth=-1, effectiveDepth=-1, effectiveHeight=-1), label='140201', color='#607D8B', weightIn_kg=387.82, floorArea_m2=1.238504, volume_m3=1.43047212, priority=0, serviceLevel=1, unitsInPallet=2160, isPartialPallet=False, fillPct=1.0, loadedToContainer=False, rejectionReason='', unloadSequence=0, destinationStop=0, isFragile=False, isHazmat=False, temperatureRequirement='ambient')

In [35]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

## Create Links for the following
1. Transport equipment assignment 🚚
   1. (transport_asset_df + load_equipment) [🛻+📦 record] 
2. 